In [1]:
from algoviz.svg import SVGView, Text, Rect, Circle
from algoviz import AlgoViz
import random # Zombies erscheinen random

from game_objects import Zombie, FastZombie, Plantie, Bullet

CELL_SIZE = 40 

WIDTH = 5   
HEIGHT = 10 

PIXEL_WIDTH = WIDTH * CELL_SIZE # Spielfeldraster 5x10
PIXEL_HEIGHT = HEIGHT * CELL_SIZE
GOAL_ZOMBIES = 10 # Zombies umlegen um zu gewinnen

view = SVGView(PIXEL_WIDTH, PIXEL_HEIGHT + 40, "Planties vs. Zombies ENDVERSION")
# +40 wegen dem Fenster oben

# Fenster: mit Text und Balken dahinter oben 
header = Rect(0, 0, PIXEL_WIDTH, 40, view)
header.set_fill("lightgray")
score_display = Text(10, 30, "Score: 0", view)
score_display.set_color("red")

# Gitter um 40 Pixel nach unten schieben wegen Fenster (y+40)
grid = 40
for r in range(HEIGHT):
    for c in range(WIDTH):
        x = c * CELL_SIZE
        y = r * CELL_SIZE + grid
        bg_rect = Rect(x, y, CELL_SIZE, CELL_SIZE, view)

        if (r + c) % 2 == 0: # c + r Entscheidung über grün (gerade) und pink (ung)
            bg_rect.set_fill("#ccffcc") 
        else:
            bg_rect.set_fill("pink")   
            
        bg_rect.set_color("black") 

# Listen/container für alle aktiven Objekte Speichern...
# wenn z.B Zombie stirbt, dann löschen aus Liste
zombies = []
planties = []
bullets = []

print("GAME STARTED!")
print("the first 2 plants are GOLD (superplanties), whereas the rest is green")

# setup
game_over = False
score = 0
timer = 0 

# Mauskontrolle
start_click = view.last_click() # wo als letztes geklickt?
last_x = start_click.x() #jeweiligen Koordinaten (wichtig für Placement der Planties)
last_y = start_click.y()

while not game_over:
    
    # PLANTIES >>>>>>>>
    current_click = view.last_click() # vgl. für neuen klick 
    curr_x = current_click.x()
    curr_y = current_click.y()
    
    if curr_x != last_x or curr_y != last_y: #Bewegt sich Maus?
        last_x = curr_x # Koordinaten
        last_y = curr_y
        
        game_y = curr_y - grid # Verschiebung durch Fenster für Raster (Zeile)
        column = int(curr_x / CELL_SIZE)  
        row = int(game_y / CELL_SIZE)

        if column >= 0 and column < WIDTH and len(planties) < WIDTH: # Klick im Feld?
            if row == HEIGHT - 1: # wurde in unterste Zeile geklickt?
                already_occupied = False
                for p in planties:
                    if p.column == column and p.row == row:
                        already_occupied = True # prüft existenz der Pflanze 

                if not already_occupied:
                    
                    # die ersten 2 Pflanzen sind und MÜSSEN SUPER (Gold)
                    make_super = False
                    if len(planties) < 2:
                        make_super = True
                    
                    #  'is_super' an die Klasse
                    new_plantie = Plantie(column, row, view, is_super=make_super)
                    
                    # Position korrigieren wegen Header
                    new_plantie.y += grid
                    new_plantie.form.move_to(int(new_plantie.x), int(new_plantie.y))
                    
                    planties.append(new_plantie) # in Liste hinzufügen 
                    
                    if make_super:
                        print(f"SUPER-Plantie ready! (column {column})")
                    else:
                        print(f"Normal Plantie ready. (column {column})")

    # ZOMBIES >>>>>>>
    if timer % 80 == 0: # "randomness Frame Zombies erstellen
        if random.random() < 0.8:
            count = random.randint(1, 2) # random ob 1 oder 2 spawnen
            for i in range(count):
                spawn_col = random.randint(0, WIDTH - 1) # wenn z.B count 2, dann 2Z
                # und random Bahn
                new_z = None
                if random.random() < 0.1: # 10% Chance auf final boss
                    new_z = FastZombie(spawn_col, 0, view)
                else:
                    new_z = Zombie(spawn_col, 0, view)
                
                new_z.y += grid
                new_z.form.move_to(int(new_z.x), int(new_z.y))
                zombies.append(new_z)

    # Bullets + Planties Zusammenarbeit >>>>>>
    for p in planties:
        p.load_gun() 
        should_shoot = False
        for z in zombies:
            if z.column == p.column and z.y < p.y:
                should_shoot = True # wann soll PLantie shooten? wenn Zombie da ist
        
        if should_shoot and p.shoot_questionmark():
            bullets.append(p.shoot()) # Waffe wird geladen und Kugel kommt in liste Bullet

    for b in bullets:
        b.move() # tatsächliche Bewegung der Kugeln
    
    for z in zombies[:]:
        is_eating = False
        for p in planties[:]:
            if z.column == p.column and abs(z.y - p.y) < 30: # hier wird Pflanze gegessen
                is_eating = True
                p.damage() 
                if p.hp <= 0:
                    p.delete() # Plantie tot
                    if p in planties: planties.remove(p) # von Liste entfernen
                    print("plantie eaten!")
        
        if not is_eating:
            z.move() # sonst soll der sich weiterhin Bewegen

        if z.y > PIXEL_HEIGHT + grid: # wenn er aus dem Bild läuft, dann vorbei
            print("GAME OVER")
            game_over = True

    active_bullets = []
    for b in bullets:
        if b.y < grid: b.delete() # Kugeln sollen gelöscht werden die aus Spielfeld sind
        else: active_bullets.append(b)
    bullets = active_bullets

    for b in bullets[:]:
        hit = False
        for z in zombies[:]:
            if b.column == z.column:
                if abs(b.y - z.y) < 20: # bei dem Abstand soll Zombie damagen
                    z.damage()
                    hit = True
                    b.delete() # wenn Zombie trifft dann verschwinden
                    if b in bullets: bullets.remove(b)
                    
                    if z.hp <= 0: # Zombie tot, Score höher machen
                        z.delete()
                        if z in zombies: zombies.remove(z)
                        score += 1
                        score_display.set_text(f"Score: {score}/{GOAL_ZOMBIES}")
                        print(f"Zombie defeated! Score: {score}")
    
    if score >= GOAL_ZOMBIES:
        print("YOU WIN!")
        game_over = True # wenn genug Punkte, dann Schleife beendet
        
    if view.last_key() != "":  # wenn Nutzer Taste drückt, Spiel beendet
        print("interruption by user.")
        game_over = True
        
    timer += 1 # spawnen
    AlgoViz.sleep(20) # etwas flüssigere Animation

print("GAME ENDS.")

GAME STARTED!
the first 2 plants are GOLD (superplanties), whereas the rest is green
SUPER-Plantie ready! (column 4)
SUPER-Plantie ready! (column 2)
Zombie defeated! Score: 1
Zombie defeated! Score: 2
Normal Plantie ready. (column 1)
Zombie defeated! Score: 3
Normal Plantie ready. (column 3)
plantie eaten!
GAME OVER
GAME ENDS.


In [2]:
!zip abgabe.zip tower_defense.ipynb game_objects.py

  adding: tower_defense.ipynb (deflated 69%)
  adding: game_objects.py (deflated 66%)
